In [1]:
import os
import platform

In [4]:
platform.platform()[0]

'W'

In [15]:
class Binary():
    """
    Binary Class
    """
    def decimal_to_binary(self, decimal_num):
        """
        This function converts base 10 to base 2
        """
        binary_num = ""
        while decimal_num > 0:
            binary_num = str(decimal_num % 2) + binary_num
            decimal_num = decimal_num // 2
        # Fill to 16 digits with 0
        if len(binary_num) < 16:
            binary_num = "0"*(16-len(binary_num)) + binary_num
        return binary_num
        
    def binary_to_decimal(self, binary_num):
        """
        This function converts base 2 to base 10
        """
        decimal_num = 0
        for i in range(len(binary_num)):
            decimal_num += int(binary_num[i]) * (2 ** (len(binary_num)-i-1))
        return decimal_num
    
    def binary_crop(self, digit, binary_num):
        """
        This function crops the last n digits of the binary number
        """
        return binary_num[len(binary_num)-digit:]

    def binary_twos_complement(self, number):
        """
        This functions converts the (negative) number to its 16-bit two's complement representation
        """
        if number < 0:
            number = (1 << 16) + number  # Adding 2^16 to the negative number
        return number
    
    def binary_reverse_twos_complement(self, number):
        """
        This functions converts the 16-bit two's complement number back to its original signed representation 
        """
        if number & (1 << 15):  # Check if the most significant bit is 1
            number = number - (1 << 16)  # Subtract 2^16 from the number
        return number

In [25]:
binary = Binary()

aaa = 4321

a = binary.binary_twos_complement(-aaa)
print(a)  # Example usage

b = binary.binary_reverse_twos_complement(a) / 10.0
print(b)

# print(binary.binary_reverse_twos_complement(0x0010))  # Example usage

61215
-432.1


# Connect Test

In [13]:
import json
from pymodbus.client import ModbusSerialClient
import time

msg = {
    "mode": "Connect",
    "action": "connect_port",
    "port": "COM3"
}

In [14]:
com_port = msg.get("port")
client = ModbusSerialClient(
    port=com_port,
    baudrate=19200,
    parity="E",     
    stopbits=1,
    bytesize=8,
    timeout=1
)

if client.connect():
    print("Connected successfully")
    msg = f"Connected to COM{com_port}!"
    sta = "success"
    client.close()
else:
    print("Connection failed")
    msg = f"Can't connect to COM{com_port}"
    sta = "failed"
    client.close()

response_data = {
    "mode": "Connect",
    "message": msg,
    "status": sta
}

print(response_data)

Connected successfully
{'mode': 'Connect', 'message': 'Connected to COMCOM3!', 'status': 'success'}


# Read & Write as rroutine

In [16]:
a = 0
if not client.connect():
    print("Connection failed")
else:
    print("Connected")

    try:
        while True:
            # Read holding registers
            result = client.read_holding_registers(
                address=0,
                count=7,        # address 0, 1, 2, 3, 4, 5, 6
                slave=21
            )
            msg = {
                "mode": "STATS",
                "pos" : result.registers[0] if not result.isError() else None,
                "vel" : result.registers[1] if not result.isError() else None,
                "acc" : result.registers[2] if not result.isError() else None,
                "grip1" : result.registers[3] if not result.isError() else None,
                "grip2" : result.registers[4] if not result.isError() else None,
                "current" : result.registers[5] if not result.isError() else None,
                "emergency" : result.registers[6] if not result.isError() else None
            }
            print(msg)
            
            # if result.isError():
            #     print("Read error:", result)
            # else:
            #     print("Read value:", result.registers)

            # Write to a holding register
            result = client.write_register(
                address=2,
                value=a & 0xFFFF,   # Write the last 16 bits of a
                slave=21
            )

            a += 1
            time.sleep(0.2)   # 0.1 Hz
    except KeyboardInterrupt:
        print("Stopped by user")

    # client.close()

Connected
{'mode': 'STATS', 'pos': 1111, 'vel': 28208, 'acc': 0, 'grip1': 0, 'grip2': 0, 'current': 0, 'emergency': 0}
{'mode': 'STATS', 'pos': 1111, 'vel': 42598, 'acc': 0, 'grip1': 0, 'grip2': 0, 'current': 0, 'emergency': 0}
{'mode': 'STATS', 'pos': 1111, 'vel': 55911, 'acc': 1, 'grip1': 0, 'grip2': 0, 'current': 0, 'emergency': 0}
{'mode': 'STATS', 'pos': 1111, 'vel': 4174, 'acc': 2, 'grip1': 0, 'grip2': 0, 'current': 0, 'emergency': 0}
{'mode': 'STATS', 'pos': 1111, 'vel': 18340, 'acc': 3, 'grip1': 0, 'grip2': 0, 'current': 0, 'emergency': 0}
{'mode': 'STATS', 'pos': 1111, 'vel': 32439, 'acc': 4, 'grip1': 0, 'grip2': 0, 'current': 0, 'emergency': 0}
Stopped by user


In [9]:
import asyncio
server_state = {
    "client": None,
    "connected": False,
    "port": None,
    "slave": 1,
}


In [12]:
old = server_state["client"]
print(old)

None


In [ ]:
# async def stats_loop(websocket):
#     last_read_time = None

#     try:
#         while True:

#             if protocol.client and protocol.is_connected():
#                 try:
#                     t_start = time.perf_counter()

#                     rr = await asyncio.to_thread(
#                         protocol.client.read_holding_registers,
#                         0, 10,
#                         slave=server_state["slave"]
#                     )

#                     t_end = time.perf_counter()
#                     print(rr)
#                     print("Registers:", rr.registers)

#                     if rr and (not rr.isError()) and hasattr(rr, "registers"):
#                         now = t_end

#                         if last_read_time is not None:
#                             dt = now - last_read_time

#                         last_read_time = now

#                         print("Registers:", rr.registers)

#                         if len(rr.registers) >= 3:
#                             # robot_state["position"] = rr.registers[0]
#                             robot_state["speed"]    = rr.registers[1]
#                             robot_state["accel"]    = rr.registers[2]

#                             # robot_state["mode"] = "Auto" if rr.registers[3] == 1 else "Manual"
#                             # robot_state["emergency"] = "Yes" if rr.registers[4] == 1 else "No"
#                             # robot_state["gripper_z"] = "Down" if rr.registers[5] == 1 else "Up"
#                             # robot_state["gripper_jaw"] = "Close" if rr.registers[6] == 1 else "Open"

#                         # print(f"dt between Modbus responses: {dt:.4f} s  ({1/dt:.2f} Hz)")
#                         # print(f"Read latency: {(t_end - t_start)*1000:.2f} ms")
#                     protocol.write_heartbeat()
#                 except Exception as e:
#                     print(f"Error reading Modbus registers: {e}")

#             payload = {
#                 "type": "STATS",
#                 "pos": robot_state["position"],
#                 "speed": robot_state["speed"],
#                 "accel": robot_state["accel"],
#                 "gripper": f"{robot_state['gripper_z']} / {robot_state['gripper_jaw']}",
#                 "mode": robot_state["mode"],
#                 "emergency": robot_state["emergency"]
#             }

#             try:
#                 await websocket.send(json.dumps(payload))
#             except ConnectionClosed:
#                 break

#             await asyncio.sleep(0.1)  # IMPORTANT: control polling rate

#     except asyncio.CancelledError:
#         pass

In [7]:
a = 0
b = 1

if a and not b:
    print(555)
    
if not a and b:
    print(666)    

666


In [11]:
if a:
    print(4)

In [13]:
if a != b:
     print("Up") if b else print("down")

Up


In [31]:
a = '11'
b = '+'

c = int(b+a)

In [32]:
c + int(a)

22

In [35]:
0x34 + 1

53